In [1]:
from pathlib import Path
import fitz
import numpy as np, pandas as pd
import matplotlib, matplotlib.pyplot as plt

plt.rcParams.update({
    "font.family": "sans-serif", "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 7, "axes.titlesize": 7, "axes.labelsize": 7, "xtick.labelsize": 6, "ytick.labelsize": 6,
    "legend.fontsize": 6, "axes.linewidth": 0.5, "xtick.major.width": 0.5, "ytick.major.width": 0.5,
    "xtick.major.size": 2.5, "ytick.major.size": 2.5, "axes.spines.top": False, "axes.spines.right": False,
    "legend.frameon": False, "pdf.fonttype": 42, "axes.titlelocation": "left"})
mm = 1 / 25.4
Path("figures").mkdir(exist_ok=True)
paper, si = "../data/raw/papers/mercado2009.pdf", "../data/raw/papers/mercado2009_SI.pdf"
sites = {"US-Bi2": "maize", "US-Ne1": "maize", "US-Ne2": "maize", "US-Ne3": "maize",
         "US-xUN": "forest", "US-UMB": "forest", "US-MMS": "forest", "US-Ha1": "forest"}
gpp_global = 123.0

mercado et al. 2009 si fig. s10: diffuse fraction contribution to gpp in 1980 and 1999, one coloured rectangle per jules grid cell (3.75 x 2.5 deg), colour class to g c m-2 yr-1

In [2]:
colours = [(0.122, 0.239, 0.98), (0.0, 0.631, 0.902), (1.0, 1.0, 1.0), (0.0, 0.859, 0.0), (0.902, 0.859, 0.2),
           (0.902, 0.69, 0.18), (0.941, 0.51, 0.161), (0.941, 0.0, 0.0), (0.859, 0.0, 0.388)]
edges = np.array([-40, -20, -10, 10, 20, 40, 60, 70, 80, 100.])
frames = {1980: (72.9, 198.1, 282.3, 295.7), 1999: (310.3, 198.1, 519.7, 295.7)}
dlon, dlat = 3.75, 2.5
lons, lats = np.arange(-180 + dlon / 2, 180, dlon), np.arange(80 - dlat / 2, -60, -dlat)
fills = [(g["rect"], tuple(round(c, 3) for c in g["fill"])) for g in fitz.open(si)[8].get_drawings() if g.get("fill")]
cells = []
for year, (x0, y0, x1, y1) in frames.items():
    cx, cy = x0 + (lons + 180) * (x1 - x0) / 360, y0 + (80 - lats) * (y1 - y0) / 140
    for r, col in fills:
        if col in colours and r.x0 > x0 - 0.5 and r.x1 < x1 + 0.5 and r.y0 > y0 - 0.5 and r.y1 < y1 + 0.5:
            cells += [(year, la, lo, colours.index(col)) for lo in lons[(cx > r.x0) & (cx < r.x1)]
                      for la in lats[(cy > r.y0) & (cy < r.y1)]]
cells = pd.DataFrame(cells, columns=["year", "lat", "lon", "cls"]).drop_duplicates(["year", "lat", "lon"], keep="last")
area = 6.371e6 ** 2 * np.radians(dlon) * (np.sin(np.radians(cells.lat + dlat / 2)) - np.sin(np.radians(cells.lat - dlat / 2)))
lo, hi = edges[:-1], edges[1:]
gpp = pd.DataFrame({y: [(area[g.index] * v[g.cls]).sum() / 1e15 for v in ((lo + hi) / 2, lo, hi)]
                    for y, g in cells.groupby("year")}, index=["mid", "lo", "hi"]).T
gpp

Out[0]: 
           mid        lo        hi
1980  1.433259  0.227822  2.638696
1999  1.327217  0.095930  2.558503


           mid        lo        hi
1980  1.433259  0.227822  2.638696
1999  1.327217  0.095930  2.558503

mercado et al. 2009 fig. 3a: global mean diffuse fraction of par, read from the vector drawing; change vs 1901-1910

In [3]:
a = fitz.open(paper)[2].get_drawings()[2]["items"]
p = np.array([[q.x, q.y] for q in [it[1] for it in a if it[0] == "l"] + [a[-1][2]]])
year = 1900 + (p[:, 0] - 81.1) / (281.8 - 81.1) * 100
df = np.interp(np.arange(1901, 2000), year, 0.40 + (143.9 - p[:, 1]) / (143.9 - 59.2) * 0.10)
df = pd.Series(df, index=np.arange(1901, 2000))
ddf = {y: df[y] - df.loc[1901:1910].mean() for y in (1980, 1999)}
ddf

Out[0]: {1980: np.float64(0.02847462343330004), 1999: np.float64(0.0371054254638149)}


{1980: np.float64(0.02847462343330004), 1999: np.float64(0.0371054254638149)}

gpp change per 0.01 increase in diffuse fraction: jules, our sites (diffuse effect / mean smoke - reference df), gu et al. 2003 (23 % noon increase / change in shortwave df at 70 deg from their fig. 1)

In [4]:
rows = []
for y in (1980, 1999):
    pct = 100 * gpp.loc[y] / gpp_global
    rows.append({"source": f"JULES {y}", "group": "mercado", "pct": pct.mid, "lo": pct.lo, "hi": pct.hi, "ddf": ddf[y]})
bars = pd.read_csv("results/bar_effects.csv").set_index(["site", "component"])
for site, group in sites.items():
    d = pd.read_csv(f"data/site_{site}.csv", parse_dates=["date"])
    ref, dd = d[d.is_bg], []
    for r in d[d.is_smoke].itertuples():
        w = ref[((ref.date - r.date).dt.days.abs() <= 15) & ((ref.hour - r.hour).abs() <= 1)]
        if len(w) >= 3:
            dd.append(r.DF - w.DF.mean())
    e = bars.loc[(site, "diffuse")]
    rows.append({"source": site, "group": group, "pct": e.pct, "lo": e.lo, "hi": e.hi, "ddf": np.mean(dd)})
el = np.sin(np.radians(70))
gu = {k: dif / (dni * el + dif) for k, (dni, dif) in {"normal": (939, 76), "1992": (893, 140)}.items()}
rows.append({"source": "Gu et al. 1992", "group": "gu", "pct": 23, "lo": np.nan, "hi": np.nan, "ddf": gu["1992"] - gu["normal"]})
t = pd.DataFrame(rows)
for k in ("pct", "lo", "hi"):
    t[k + "_per001"] = t[k] / t.ddf / 100
t.round(4).to_csv("results/mercado_sensitivity.csv", index=False)
t.round(3)

Out[0]: 
            source    group     pct  ...  pct_per001  lo_per001  hi_per001
0       JULES 1980  mercado   1.165  ...       0.409      0.065      0.753
1       JULES 1999  mercado   1.079  ...       0.291      0.021      0.561
2           US-Bi2    maize   4.644  ...       0.394      0.029      0.774
3           US-Ne1    maize  15.596  ...       0.985      0.644      1.380
4           US-Ne2    maize  17.880  ...       1.113      0.689      1.620
5           US-Ne3    maize  11.731  ...       0.687      0.301      1.116
6           US-xUN   forest  16.716  ...       1.012      0.637      1.514
7           US-UMB   forest   4.933  ...       0.429      0.139      0.747
8           US-MMS   forest   7.748  ...       0.367      0.217      0.540
9           US-Ha1   forest   8.877  ...       0.567      0.293      0.900
10  Gu et al. 1992       gu  23.000  ...       3.612        NaN        NaN

[11 rows x 9 columns]


            source    group     pct  ...  pct_per001  lo_per001  hi_per001
0       JULES 1980  mercado   1.165  ...       0.409      0.065      0.753
1       JULES 1999  mercado   1.079  ...       0.291      0.021      0.561
2           US-Bi2    maize   4.644  ...       0.394      0.029      0.774
3           US-Ne1    maize  15.596  ...       0.985      0.644      1.380
4           US-Ne2    maize  17.880  ...       1.113      0.689      1.620
5           US-Ne3    maize  11.731  ...       0.687      0.301      1.116
6           US-xUN   forest  16.716  ...       1.012      0.637      1.514
7           US-UMB   forest   4.933  ...       0.429      0.139      0.747
8           US-MMS   forest   7.748  ...       0.367      0.217      0.540
9           US-Ha1   forest   8.877  ...       0.567      0.293      0.900
10  Gu et al. 1992       gu  23.000  ...       3.612        NaN        NaN

[11 rows x 9 columns]

In [5]:
m, sink = t[t.group == "mercado"], 23.7
fig, (a1, a2) = plt.subplots(1, 2, figsize=(180 * mm, 85 * mm), gridspec_kw={"width_ratios": [1, 1.5]})
fig.subplots_adjust(left=0.08, right=0.98, top=0.9, bottom=0.22, wspace=0.3)
vals = [sink, *m.pct]
a1.bar(range(3), vals, color=["#003466", "#70a0cd", "#70a0cd"], width=0.6)
a1.errorbar([1, 2], m.pct, yerr=[m.pct - m.lo, m.hi - m.pct], fmt="none", color="black", lw=1)
for x, v in enumerate(vals):
    a1.text(x, v + 1, f"{v:.1f} %", ha="center", va="bottom")
a1.set_xticks(range(3), ["Land sink\n1960–99", "Global\nGPP 1980", "Global\nGPP 1999"])
a1.set_ylim(0, 30)
a1.set_ylabel("Diffuse-fraction effect (%)")
a1.set_title("(a) Mercado et al. (2009)")
col = {"mercado": "#70a0cd", "gu": "black", "maize": "#c47900", "forest": "#004f00"}
for x, r in enumerate(t.itertuples()):
    yerr = None if np.isnan(r.lo) else [[r.pct_per001 - r.lo_per001], [r.hi_per001 - r.pct_per001]]
    a2.errorbar(x, r.pct_per001, yerr=yerr, fmt="*" if r.group == "gu" else "o", color=col[r.group], ecolor="black",
                lw=1, ms=8 if r.group == "gu" else 5)
a2.set_xticks(range(len(t)), t.source, rotation=45, ha="right", rotation_mode="anchor")
a2.set_xlim(-0.6, len(t) - 0.4)
a2.set_ylim(0, 5)
a2.set_ylabel("GPP change per 0.01 increase\nin diffuse fraction (%)")
a2.set_title("(b) Diffuse-light sensitivity of GPP")
a2.legend(handles=[plt.Line2D([], [], marker="*" if k == "gu" else "o", ls="", color=col[k], label=l, ms=8 if k == "gu" else 5)
                   for k, l in (("mercado", "JULES (Mercado)"), ("gu", "Gu et al. (2003)"), ("maize", "Smoke, maize"),
                                ("forest", "Smoke, forest"))], loc="upper left", frameon=True, edgecolor="black")
fig.savefig("figures/fig7_mercado.png", dpi=600)